# Assignment 2 - Evaluation and Report

This notebook compares baselines, the direct bus model, and the zone-allocation model for next-day and next-month views at both bus and zone-aggregated levels.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

repo_root = Path.cwd()
for candidate in [repo_root, *repo_root.parents]:
    if (candidate / "data").exists() and (candidate / "solution" / "assignment_2" / "src").exists():
        repo_root = candidate
        break
else:
    raise FileNotFoundError("Could not find ECESIS repository root from notebook path.")

src_dir = repo_root / "solution" / "assignment_2" / "src"
outputs_dir = repo_root / "solution" / "assignment_2" / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(src_dir))

pd.set_option("display.max_columns", None)


In [ ]:
evaluation = pd.read_csv(outputs_dir / "evaluation_summary.csv")
walk_forward = pd.read_csv(outputs_dir / "walk_forward_results.csv")
direct = pd.read_csv(outputs_dir / "bus_forecast_direct.csv")
allocated = pd.read_csv(outputs_dir / "bus_forecast_zone_allocated.csv")
baselines = pd.read_csv(outputs_dir / "baseline_forecasts.csv")
evaluation.sort_values(["horizon", "level", "wmape"])

## Interpretation

Direct bus forecasting preserves measurement-location granularity, so it can learn bus-specific behavior but is noisier and more sensitive to missing or irregular bus histories. Zone forecasting is smoother because aggregation cancels some bus-level volatility; the tradeoff is that bus-share allocation can miss bus-specific deviations when shares change. Comparing both paths is valuable because operations may need bus-level predictions while model stability may be stronger at the zone level.

## Leakage Prevention

The pipeline uses chronological folds only. Rolling features are shifted before rolling, validation historical averages are fitted from the training window, lag features are masked when their source timestamp would occur after `forecast_created_at`, and 2025 is reserved as a final holdout period.

In [ ]:
per_zone_direct = direct.groupby(["horizon", "zone_name"], as_index=False).apply(
    lambda g: pd.Series({"mae": (g["predicted_pd"] - g["actual_pd"]).abs().mean()})
).reset_index(drop=True)
per_zone_direct.to_csv(outputs_dir / "per_zone_direct_error.csv", index=False)
per_zone_direct.head()

In [ ]:
per_bus_direct = direct.groupby(["horizon", "bus_id"], as_index=False).apply(
    lambda g: pd.Series({"mae": (g["predicted_pd"] - g["actual_pd"]).abs().mean()})
).reset_index(drop=True)
per_bus_direct.to_csv(outputs_dir / "per_bus_direct_error_distribution.csv", index=False)
per_bus_direct["mae"].describe().to_frame().T

## Final Selection Rationale

For this prototype, selection should be based on WMAPE and MAE by horizon and level. Baselines remain important controls: if the direct or hierarchical models do not beat previous-week same-hour behavior, the feature set or validation horizon should be revisited before scaling to the full 2022-2024 training window and 2025 test year.